In [129]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [130]:
df_2023 = pd.read_excel(fr"C:\Users\zhangbon\Desktop\2023年全年发货明细.xlsx")
df_2024 = pd.read_excel(fr"C:\Users\zhangbon\Desktop\2023-2024年每个月发货清单-定期更新-12月底.xlsx",sheet_name='周期年 (总)-金额')
df_2025 = pd.read_excel(fr"C:\Users\zhangbon\Desktop\2025发货.xlsx")
df_plm = pd.read_excel(fr"D:\000物料报表\202510\单型号贡献-低效-长尾\产品生命周期状态全表.xlsx")
df_plm['物料号'] = df_plm['物料号'].astype(str)
df_2023['物料编码'] = df_2023['物料编码'].astype(str)
df_2024['物料编码'] = df_2024['物料编码'].astype(str)
df_2025['物料编码'] = df_2025['物料编码'].astype(str)




In [131]:
df_plm = df_plm[(df_plm['物料号'].str.len() == 13)&(df_plm['国内/海外'] == '国内')&(df_plm['下属渠道'].isin(['零售','工程','电商','米博新零售','商净新零售']))&(df_plm['产品线'].isin(['油烟机产品线','烹饪厨电产品线','净热产品线','洗碗机产品线','冰储产品线']))]
df_plm['停止销售时间'] = pd.to_datetime(df_plm['停止销售时间'], format='mixed')
df_plm['开始销售时间'] = pd.to_datetime(df_plm['开始销售时间'], format='mixed')
df_plm['停止发货时间'] = pd.to_datetime(df_plm['停止发货时间'], format='mixed')
# 转为年月日格式
df_plm['停止销售时间'] = df_plm['停止销售时间'].dt.strftime('%Y-%m-%d')
df_plm['开始销售时间'] = df_plm['开始销售时间'].dt.strftime('%Y-%m-%d')
df_plm['停止发货时间'] = df_plm['停止发货时间'].dt.strftime('%Y-%m-%d')

#排除空值带来的判断影响
df_plm['开始销售时间'].fillna('2200-01-01',inplace=True)
df_plm['停止销售时间'].fillna('2200-01-01',inplace=True)
df_plm['停止发货时间'].fillna('2200-01-01',inplace=True)    

for index,row in df_plm.iterrows():
    df_plm.loc[index,'最早开始销售时间-标准型号'] = df_plm[df_plm['标准型号'] == df_plm.loc[index,'标准型号']]['开始销售时间'].min()
    df_plm.loc[index,'最晚停止销售时间-标准型号'] = df_plm[df_plm['标准型号'] == df_plm.loc[index,'标准型号']]['停止销售时间'].max()
    df_plm.loc[index,'最晚停止发货时间-标准型号'] = df_plm[df_plm['标准型号'] == df_plm.loc[index,'标准型号']]['停止发货时间'].max()
df_plm['最早开始销售时间-标准型号'] = pd.to_datetime(df_plm['最早开始销售时间-标准型号'], format='mixed')
df_plm['最晚停止销售时间-标准型号'] = pd.to_datetime(df_plm['最晚停止销售时间-标准型号'], format='mixed')
df_plm['最晚停止发货时间-标准型号'] = pd.to_datetime(df_plm['最晚停止发货时间-标准型号'], format='mixed')
# df_plm['最早开始销售时间-标准型号']
need_cols_plm = df_plm[['物料号','标准型号','最早开始销售时间-标准型号','最晚停止销售时间-标准型号','最晚停止发货时间-标准型号','产品状态','产品线']]


C:\Users\zhangbon\AppData\Local\Temp\ipykernel_4204\2128706956.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_plm['开始销售时间'].fillna('2200-01-01',inplace=True)
C:\Users\zhangbon\AppData\Local\Temp\ipykernel_4204\2128706956.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.


### 先做2023年的数据

In [132]:
df1 = pd.DataFrame()
df1['物料编码'] = df_2023['物料编码'].drop_duplicates().reset_index(drop=True)
df1 = pd.merge(df1,need_cols_plm,left_on='物料编码',right_on='物料号',how='left')
df1


,物料编码,物料号,标准型号,最早开始销售时间-标准型号,最晚停止销售时间-标准型号,最晚停止发货时间-标准型号,产品状态,产品线
0,1001000200087,1001000200087,EA06,2018-05-18,2200-01-01,2200-01-01,量产,油烟机产品线
1,1001000900383,1001000900383,EM35A,2023-04-08,2200-01-01,2200-01-01,量产,油烟机产品线
2,1001001500121,1001001500121,01-F2.i,2023-06-01,2200-01-01,2200-01-01,量产,油烟机产品线
3,1001000900286,1001000900286,EMG1,2020-07-06,2200-01-01,2200-01-01,量产,油烟机产品线
4,1001002000027,1001002000027,03-X1S-H,2023-07-15,2200-01-01,2200-01-01,量产,油烟机产品线
...,...,...,...,...,...,...,...,...
1854,1021000300001,NaN,NaN,NaT,NaT,NaT,NaN,NaN
1855,1021000100001,NaN,NaN,NaT,NaT,NaT,NaN,NaN
1856,1021000400001,NaN,NaN,NaT,NaT,NaT,NaN,NaN
1857,1021000100002,NaN,NaN,NaT,NaT,NaT,NaN,NaN


In [133]:
df_out1 = pd.DataFrame()
df_out1['产品线'] = ['油烟机产品线','烹饪厨电产品线','净热产品线','洗碗机产品线','冰储产品线']
for index,row in df_out1.iterrows():
    product_line = row['产品线']
    df_out1.loc[index,'2023年开始销售标准型号数'] = df1[(df1['产品线'] == product_line) & (df1['最早开始销售时间-标准型号'].dt.year == 2023)]['标准型号'].nunique()
    df_out1.loc[index,'2023年停止销售标准型号数'] = df1[(df1['产品线'] == product_line) & (df1['最晚停止销售时间-标准型号'].dt.year == 2023)]['标准型号'].nunique()
    df_out1.loc[index,'2023年停止生产标准型号数（停止发货）'] = df1[(df1['产品线'] == product_line) & (df1['最晚停止发货时间-标准型号'].dt.year == 2023)]['标准型号'].nunique()
df_out1


,产品线,2023年开始销售标准型号数,2023年停止销售标准型号数,2023年停止生产标准型号数（停止发货）
0,油烟机产品线,22.0,33.0,10.0
1,烹饪厨电产品线,20.0,12.0,0.0
2,净热产品线,16.0,3.0,4.0
3,洗碗机产品线,28.0,9.0,1.0
4,冰储产品线,3.0,3.0,1.0


### 2024年的数据

In [134]:
df2 = pd.DataFrame()
df2['物料编码'] = df_2024['物料编码'].drop_duplicates().reset_index(drop=True)
df2 = pd.merge(df2,need_cols_plm,left_on='物料编码',right_on='物料号',how='left')
df2


,物料编码,物料号,标准型号,最早开始销售时间-标准型号,最晚停止销售时间-标准型号,最晚停止发货时间-标准型号,产品状态,产品线
0,1001000200087,1001000200087,EA06,2018-05-18,2200-01-01,2200-01-01,量产,油烟机产品线
1,1001000300073,1001000300073,JX17S,2013-09-26,2023-08-28,2200-01-01,停止销售,油烟机产品线
2,1001000300073,1001000300073,JX17S,2013-09-26,2023-08-28,2200-01-01,停止销售,油烟机产品线
3,1001000300073,1001000300073,JX17S,2013-09-26,2023-08-28,2200-01-01,停止销售,油烟机产品线
4,1001000500073,1001000500073,JQ01TS,2014-12-24,2023-12-14,2200-01-01,停止销售,油烟机产品线
...,...,...,...,...,...,...,...,...
2054,1023000300048,NaN,NaN,NaT,NaT,NaT,NaN,NaN
2055,1023000300049,NaN,NaN,NaT,NaT,NaT,NaN,NaN
2056,1023000300050,NaN,NaN,NaT,NaT,NaT,NaN,NaN
2057,1023000300052,NaN,NaN,NaT,NaT,NaT,NaN,NaN


In [135]:
df_out2 = pd.DataFrame()
df_out2['产品线'] = ['油烟机产品线','烹饪厨电产品线','净热产品线','洗碗机产品线','冰储产品线']
for index,row in df_out2.iterrows():
    product_line = row['产品线']
    df_out2.loc[index,'2024年开始销售标准型号数'] = df2[(df2['产品线'] == product_line) & (df2['最早开始销售时间-标准型号'].dt.year == 2024)]['标准型号'].nunique()
    df_out2.loc[index,'2024年停止销售标准型号数'] = df2[(df2['产品线'] == product_line) & (df2['最晚停止销售时间-标准型号'].dt.year == 2024)]['标准型号'].nunique()
    df_out2.loc[index,'2024年停止生产标准型号数（停止发货）'] = df2[(df2['产品线'] == product_line) & (df2['最晚停止发货时间-标准型号'].dt.year == 2024)]['标准型号'].nunique()
df_out2


,产品线,2024年开始销售标准型号数,2024年停止销售标准型号数,2024年停止生产标准型号数（停止发货）
0,油烟机产品线,21.0,14.0,1.0
1,烹饪厨电产品线,33.0,23.0,1.0
2,净热产品线,24.0,15.0,0.0
3,洗碗机产品线,29.0,22.0,0.0
4,冰储产品线,12.0,2.0,0.0


### 2025年数据

In [136]:
df3 = pd.DataFrame()
df3['物料编码'] = df_2025['物料编码'].drop_duplicates().reset_index(drop=True)
df3 = pd.merge(df3,need_cols_plm,left_on='物料编码',right_on='物料号',how='left')
df3


,物料编码,物料号,标准型号,最早开始销售时间-标准型号,最晚停止销售时间-标准型号,最晚停止发货时间-标准型号,产品状态,产品线
0,1009001100002,1009001100002,ZK50-01-F1.i,2022-01-04,2025-08-01,2200-01-01,停止销售,烹饪厨电产品线
1,1009001100002,1009001100002,ZK50-01-F1.i,2022-01-04,2025-08-01,2200-01-01,停止销售,烹饪厨电产品线
2,1009001100002,1009001100002,ZK50-01-F1.i,2022-01-04,2025-08-01,2200-01-01,停止销售,烹饪厨电产品线
3,1001002100022,1001002100022,JC03A,2021-05-17,2200-01-01,2200-01-01,量产,油烟机产品线
4,1001002100022,1001002100022,JC03A,2021-05-17,2200-01-01,2200-01-01,量产,油烟机产品线
...,...,...,...,...,...,...,...,...
2354,1004001100001,1004001100001,L1PB26-P03,2021-02-28,2200-01-01,2200-01-01,退市预警,净热产品线
2355,1023000200004,NaN,NaN,NaT,NaT,NaT,NaN,NaN
2356,1001000900428,1001000900428,HE1-G,2025-09-25,2200-01-01,2200-01-01,量产,油烟机产品线
2357,1008000600007,1008000600007,JBSD2F-02-M5,2025-09-15,2200-01-01,2200-01-01,量产,洗碗机产品线


In [137]:
df_out3 = pd.DataFrame()
df_out3['产品线'] = ['油烟机产品线','烹饪厨电产品线','净热产品线','洗碗机产品线','冰储产品线']
for index,row in df_out3.iterrows():
    product_line = row['产品线']
    df_out3.loc[index,'2025年开始销售标准型号数'] = df3[(df3['产品线'] == product_line) & (df3['最早开始销售时间-标准型号'].dt.year == 2025)]['标准型号'].nunique()
    df_out3.loc[index,'2025年停止销售标准型号数'] = df3[(df3['产品线'] == product_line) & (df3['最晚停止销售时间-标准型号'].dt.year == 2025)]['标准型号'].nunique()
    df_out3.loc[index,'2025年停止生产标准型号数（停止发货）'] = df3[(df3['产品线'] == product_line) & (df3['最晚停止发货时间-标准型号'].dt.year == 2025)]['标准型号'].nunique()
df_out3


,产品线,2025年开始销售标准型号数,2025年停止销售标准型号数,2025年停止生产标准型号数（停止发货）
0,油烟机产品线,26.0,16.0,0.0
1,烹饪厨电产品线,36.0,43.0,16.0
2,净热产品线,12.0,4.0,3.0
3,洗碗机产品线,32.0,21.0,0.0
4,冰储产品线,3.0,1.0,0.0


In [138]:
with pd.ExcelWriter(r"C:\Users\zhangbon\Desktop\20232425年数据统计-胡维益.xlsx") as writer:
    df1.to_excel(writer, sheet_name='2023发货匹配PLM', index=False)
    df2.to_excel(writer, sheet_name='2024发货匹配PLM', index=False)
    df3.to_excel(writer, sheet_name='2025发货匹配PLM', index=False)
    df_out1.to_excel(writer, sheet_name='2023年数据统计', index=False)
    df_out2.to_excel(writer, sheet_name='2024年数据统计', index=False)
    df_out3.to_excel(writer, sheet_name='2025年数据统计', index=False)